In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

import os

figure_dir = "figures/revision/supplement"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
from tqdm.auto import tqdm

from spatial_tcr.clonal_expansion import (
    calc_empirical_p_values,
)

In [ ]:
def entropy(probs, axis=1):
    zero_mask = probs == 0
    probs_mod = probs.copy()
    probs_mod[zero_mask] = 1e-10
    H = -np.sum(probs * np.log2(probs_mod), axis=axis)
    return H

In [ ]:
data_dir = "data/xenium/processed"
path = os.path.join(data_dir, "08.1-kidney_tcr_clonal_clusters.h5ad")
adata = sc.read_h5ad(path)
adata

In [ ]:
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()
ad_cluster = adata[adata.obs["avbv_cluster"].notna()].copy()
ad_cluster

In [ ]:
ct_key = "cell_type_l1.1"

In [ ]:
ad_t.obs[ct_key].value_counts()

In [ ]:
df = (
    ad_cluster.obs.groupby("avbv_cluster", observed=True)[ct_key]
    .value_counts(normalize=True)
    .unstack()
)
assert df.sum(axis=1).min() == 1
H_obs = entropy(df.values)
print(f"mean entropy: {H_obs.mean()}")

In [ ]:
ad_cluster.obs[ct_key].value_counts()

In [ ]:
H_perm_values = []
n_perms = 1000
for _ in tqdm(range(n_perms)):
    # permute tcell_subtype labels
    ad_cluster_perm = ad_cluster.copy()
    ad_cluster_perm.obs[ct_key] = ad_cluster_perm.obs[ct_key].sample(frac=1).values
    df_perm = (
        ad_cluster_perm.obs.groupby("avbv_cluster", observed=True)[ct_key]
        .value_counts(normalize=True)
        .unstack()
    )
    assert np.abs(df_perm.sum(axis=1).min() - 1) < 1e-10
    H_perm = entropy(df_perm.values)
    H_perm_values.append(H_perm)

H_perm_mean_values = np.array(H_perm_values).mean(axis=1)
# H_perm_mean_values

In [ ]:
df_perm.sum(axis=1).min()

In [ ]:
n_perms = 1000
H_sim = H_perm_mean_values
H_obs = H_obs.mean()
z_scores, p_values, q_values = calc_empirical_p_values(H_obs, H_sim, n_perms)

In [ ]:
sns.set_theme(style="ticks", context="paper")
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(H_sim, ax=ax)
ax.set_xlabel("entropy score")
ax.set_ylabel("count")
ax.set_title("average entropy of T cell subtype distribution across clonal clusters")

# draw red vertical line for observed number of clonal clusters
ax.axvline(H_obs, color="red", linestyle="--")
# add text for observed number of clonal clusters
ax.text(
    H_obs + 0.0025,
    105,
    f"Observed average entropy: {np.round(H_obs, 3)}\nPermutation p-value: {np.round(p_values, 5)}",
    color="red",
    ha="left",
)

# split x axis into two parts
sns.despine(ax=ax)

plt.savefig(
    os.path.join(figure_dir, "cc_ct_composition_entropy.pdf"),
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)